In [ ]:
import pickle

# pick whichever model scored better on ROC-AUC
best_model = gb_clf if roc_auc_score(y_test, y_proba_gb) > roc_auc_score(y_test, y_proba) else logreg

with open("../models/fit_predictor.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("../models/fit_predictor_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved fit_predictor.pkl")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)  # tree models don't need scaling

y_pred_gb = gb_clf.predict(X_test)
y_proba_gb = gb_clf.predict_proba(X_test)[:, 1]

print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Gradient Boosting ROC-AUC:", roc_auc_score(y_test, y_proba_gb))
print(classification_report(y_test, y_pred_gb, zero_division=0))

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, classification_report, confusion_matrix
)

y_pred = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred, zero_division=0))
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Only include numerical features for scaling and the logistic regression model
feature_cols = ["similarity"]
X = pairs_df[feature_cols]
y = pairs_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train_scaled, y_train)

In [ ]:
def generate_pairs(resumes_df, job_corpus, job_tfidf, job_vectors,
                    n_resumes=300, n_pos=3, n_neg=3, random_state=42):
    rng = np.random.default_rng(random_state)
    # Ensure n_resumes does not exceed the actual number of resumes available
    actual_n_resumes = min(n_resumes, len(resumes_df))
    sample_resumes = resumes_df.sample(n=actual_n_resumes, random_state=random_state)

    rows = []
    for _, r in sample_resumes.iterrows():
        cleaned = clean_text(r["Resume"])
        resume_vec = job_tfidf.transform([cleaned])
        sims = cosine_similarity(resume_vec, job_vectors).flatten()

        top_idx = sims.argsort()[::-1][:n_pos]
        bottom_pool = sims.argsort()[: len(sims) // 2]
        neg_idx = rng.choice(bottom_pool, size=n_neg, replace=False)

        for idx, label in list(zip(top_idx, [1] * n_pos)) + list(zip(neg_idx, [0] * n_neg)):
            job_row = job_corpus.iloc[idx]
            # build_feature_row is not defined in this scope. Placeholder for demonstration.
            # For a proper run, build_feature_row should be imported or defined.
            features = {"resume_text": r["Resume"], "job_title": job_row["title"], "similarity": sims[idx]}
            # features = build_feature_row(r["Resume"], job_row, sims[idx]) # Original line
            features["label"] = label
            rows.append(features)

    return pd.DataFrame(rows)

# Adjust n_resumes to match the available data
pairs_df = generate_pairs(resumes, job_corpus, job_tfidf, job_vectors, n_resumes=len(resumes))
print(pairs_df.shape)
print(pairs_df["label"].value_counts())

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# Temporarily define clean_text here until modularization is fully set up
import re
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# from features.text_features import clean_text # Commented out due to ModuleNotFoundError
# from features.match_features import build_feature_row # Commented out as not used in this cell

resumes = pd.read_csv("../data/processed/resumes_clean.csv")
job_corpus = pd.read_csv("../data/processed/job_corpus_final_v2.csv", low_memory=False)

with open("../models/job_tfidf_vectorizer.pkl", "rb") as f:
    job_tfidf = pickle.load(f)
with open("../data/processed/job_vectors.pkl", "rb") as f:
    job_vectors = pickle.load(f)

print("Resumes:", resumes.shape, "Jobs:", job_corpus.shape)